In [47]:
import json
import requests

# 模拟 API Gateway URL (替换为实际 URL)
API_URL = "https://sptadsyzvugpstxq5t5mfc54yq0aifuh.lambda-url.ap-southeast-1.on.aws/"

# 通用请求发送函数
def send_redis_request(action, key=None, value=None, ttl=None):
    # 构造 QueryData
    query_data = {
        "action": action
    }
    if key is not None:
        query_data["key"] = key
    if value is not None:
        query_data["value"] = value
    if ttl is not None:
        query_data["ttl"] = ttl

    # 构造完整 Body
    payload = {
        "Cmd": "cacheService",
        "QueryData": query_data
    }

    # 构造 Headers (Lambda 强制要求)
    headers = {
        "Content-Type": "application/json",
        "Content-Hmac": "test_signature"
    }

    # print(f"\n--- Testing Action: {action} ---")
    # print(f"Payload: {json.dumps(payload, indent=2)}")
    
    # 实际发送请求 (取消注释以运行)
    try:
        resp = requests.post(API_URL, json=payload, headers=headers)
        print(f"Response: {resp.text}")
    except Exception as e:
        print(f"Request failed: {e}")

In [48]:
import pymysql
import pandas as pd

def read_from_mysql(sql=None):
    assert sql is not None, "sql must be provided"
    # 数据库连接配置
    config = {
        'host': '10.0.0.9',      # 数据库地址
        'port': 3306,             # 端口
        'user': 'bigquery_user',           # 用户名
        'password': 'xdf#42sdfkjdfiyuISYFj76',   # 密码
        'database': 'niuniuh5_db',    # 数据库名
        'charset': 'utf8mb4',     # 字符集
        'cursorclass': pymysql.cursors.DictCursor
    }

    try:
        # 建立连接
        connection = pymysql.connect(**config)
        res=[]
        try:
            # 方法1：使用 cursor 执行 SQL
            with connection.cursor() as cursor:
                cursor.execute(sql)
                result = cursor.fetchall()
                #print("--- Cursor Result ---")
                for row in result:
                    #print(row)
                    res.append(row)
            return pd.DataFrame(res)
        finally:
            # 关闭连接
            connection.close()
            
    except Exception as e:
        print(f"Error: {e}")

In [49]:
def write_to_redis(df):
    df_dict=df.to_dict(orient='records')
    for item in df_dict:
        key=f"{item['ngametype']}:{item['nplayerid']}"
        val=float(item['total_score'])
        send_redis_request(action="set", key=key, value=val, ttl=86400)


In [50]:
dt='2026-01-09'

In [51]:
sql_400=f"""
SELECT 
    nplayerid,
    400 AS ngametype,
    brobot,
    ROUND(SUM(player_pnl), 2) AS total_score
FROM (
    SELECT 
        pnl.nplayerid,
        u.brobot,
        pnl.player_pnl
    FROM (
        SELECT 
            t1.sztoken,
            t2.nplayerid,
            t1.create_dt,
            CASE 
                WHEN t2.reward > 0 THEN t2.reward - t1.entry_fee
                ELSE -t1.entry_fee 
            END AS player_pnl
        FROM (
            SELECT DISTINCT
                sztoken,
                szroomname,
                createtime,
                SUBSTR(createtime, 1, 10) AS create_dt,
                CAST(JSON_EXTRACT(szRuleJson, '$.joinFeeUcoin') AS UNSIGNED) / 1000000 AS entry_fee,
                CAST(JSON_EXTRACT(szRuleJson, '$.awards[0].award') AS UNSIGNED) / 1000000 AS award
            FROM table_dz_game_info
            WHERE SUBSTR(createtime, 1, 10) = '{dt}'
              AND ngametype = 400
        ) t1
        LEFT JOIN (
            SELECT 
                sztoken,
                nplayerid,
                reward / 1000000 AS reward
            FROM table_dz_match_result
            WHERE SUBSTR(tCreateTime, 1, 10) = '{dt}'
              AND ngametype = 400
        ) t2 ON t1.sztoken = t2.sztoken
    ) pnl
    LEFT JOIN table_user u ON pnl.nplayerid = u.nplayerid
    WHERE pnl.create_dt = '{dt}'
) final
GROUP BY nplayerid, brobot;
"""

In [52]:
df=read_from_mysql(sql_400)


In [53]:
write_to_redis(df)

Response: {"Res":1,"ResData":"Key '400:162691' set successfully","Msg":"Success","Delta":0.0008752346038818359}
Response: {"Res":1,"ResData":"Key '400:188209' set successfully","Msg":"Success","Delta":0.0007965564727783203}
Response: {"Res":1,"ResData":"Key '400:193084' set successfully","Msg":"Success","Delta":0.012622833251953125}
Response: {"Res":1,"ResData":"Key '400:209466' set successfully","Msg":"Success","Delta":0.0009005069732666016}


In [31]:
df

,nplayerid,ngametype,brobot,total_score
0,162691,400,0,0.26
1,188209,400,1,-0.10
2,193084,400,0,-0.10
3,209466,400,0,-0.10
